In [11]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm, uniform
import plotly.graph_objects as go
N = 12376

def exp( x , a, A, tau ):
    return a*N*expon.cdf(x , A , tau) 


def exp_unif(x, a, b, tau, A):
    return a * N * (expon.cdf(x, A, tau) + b * N * uniform.cdf(x, 0, 8e-6))

# def exp_gauss_cdf(N_exp , A , tau):
#     def fixed_exp( x , N , mu , sigma):
#         return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
#     return fixed_exp
    
# def gauss_exp_cdf(N_g , mu , sigma):
#     def fixed_gauss( x , N , A , tau):
#         return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
#     return fixed_gauss

def exp_fit(cdf, a, A, tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, A, tau )
    n.fixed["A"] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def exp_unif_fit(cdf, a, b, tau, A, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, b, tau, A )
    # n.fixed['N'] = True
    n.fixed['A'] = True
    # n.limits['b'] = (0, 1)
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

In [2]:
data_0 = np.genfromtxt("Data/timestamp/21_01_2024_17_42.csv", delimiter=',')
data_1 = np.genfromtxt("Data/timestamp/23_01_2026_17_31.csv", delimiter=',')
data = np.concatenate((data_0, data_1))
n_bins = 80
# data_2 = data_2[(data_2 > 0.6e-6) & (data_2 < 1.8e-6)]
count, edges = np.histogram(data, bins=n_bins , density=False)
count_1, edges_1 = np.histogram(data_1, bins=n_bins , density=False)
print (len(data), len(data_1))
fig = go.Figure()

# fig.add_trace(go.Bar(x=edges[:-1], y=count, name='Histogram', width=np.diff(edges)))
fig.add_trace(go.Bar(x=edges[:-1], y=count, name='total', width=np.diff(edges)))
fig.add_trace(go.Bar(x=edges_1[:-1], y=count_1, name='only today', width=np.diff(edges_1)))

fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

12376 9400


In [3]:
k = exp_fit( exp, 1, 0, 2.2e-6, count_1 , edges_1)
display(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 232.2 (χ²/ndof = 3.0)      │              Nfcn = 53               │
│ EDM = 4.58e-12 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   0.780   │   0.008   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.231e-6  │ 0.029e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │           a           A         tau │
├─────┼─────────────────────────────────────┤
│   a │    6.57e-05           0 28.4086e-12 │
│   A │           0           0           0 │
│ tau │ 28.4086e-12           0    8.34e-16 │
└─────┴─────────────────────────────────────┘

In [17]:
data1 = data[data > 8.5e-7]
count_2, edges_2 = np.histogram(data1, bins =n_bins , density=False)
n = exp_fit( exp, *k.values, count_2 , edges_2)
display(n)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 228 (χ²/ndof = 2.9)        │              Nfcn = 50               │
│ EDM = 1.79e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.036   │   0.012   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.197e-6  │ 0.032e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬───────────────────────────────────────────┐
│     │             a             A           tau │
├─────┼───────────────────────────────────────────┤
│   a │      0.000144             0 -125.1735e-12 │
│   A │             0             0         0e-15 │
│ tau │ -125.1735e-12         0e-15      1.01e-15 │
└─────┴───────────────────────────────────────────┘